In [778]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels
%pip install linearmodels
%pip install stargazer

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import matplotlib.pyplot as plt


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [988]:
################ Import data
df = pd.read_csv('model_ready.csv')
supply_df = pd.read_csv('supply_ready.csv')
df = df.reset_index(drop=True)

# 将 supply_df 中的 road_fuel_IV 合并到 df
# 假设合并键为 province 和 year

df = df.merge(
    supply_df[['province', 'year', 'road_fuel_IV', 'num_models_in_market']],
    on=['province', 'year'],
    how='left'
)


# Set range, log_charging_stock, log_charging_IV, and battery_capacity to 0 for non-EVs
df.loc[df['is_electric'] == 0, ['range', 'battery_capacity']] = 0



# Recalculate log transformations with adjusted shares
df['log_sjm'] = np.log(df['shares'])
df['log_s0m'] = np.log(1 - df.groupby('market_ids')['shares'].transform('sum'))
df['log_sj_g'] = np.log(df['shares'] / df.groupby(['market_ids', 'nesting_ids'])['shares'].transform('sum'))
df['log_charging_stock'] = np.log(df['charging_stations_stock'])



In [949]:
# Define list of IVs
iv_list = [
    'cost_shifter', 
    'product_set_size',
    'euclidean_range',
    'local_range',
    'euclidean_battery',
    'local_battery',
    'euclidean_power',
    'num_models_in_market',
    'road_fuel_IV',
    'local_power',
]

iv_str = ' + '.join(iv_list)

In [989]:
################ 2SLS model using custom instruments

# First stage for charging station: regress log_charging_stock on instrument variables
X = sm.add_constant(df['log_charging_IV'])
y = df['log_charging_stock']
first_stage = sm.OLS(y, X).fit()
df['log_charging_stock_hat'] = first_stage.predict(X)

# Define the formula for the IV2SLS model
formula = f'''
(log_sjm - log_s0m) ~ 0 + is_electric*log_charging_stock_hat + range + power + battery_capacity
    + [net_prices + log_sj_g ~ {iv_str}]
'''
# CAN HAVE HORSEPOWER = POWER / MASS #

iv_model = IV2SLS.from_formula(formula, data=df).fit(cov_type="clustered", clusters=df['market_ids'])

print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                log_sjm   R-squared:                      0.9823
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9823
No. Observations:               33545   F-statistic:                 7.239e+04
Date:                Mon, Aug 04 2025   P-value (F-stat)                0.0000
Time:                        04:19:16   Distribution:                  chi2(8)
Cov. Estimator:             clustered                                         
                                                                              
                                         Parameter Estimates                                          
                                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------------------------
is_electric                           -15.658     0.6756   

In [951]:
# Output demand results as LaTeX table using Stargazer, only show selected coefficients
demand_stargazer = Stargazer([iv_model])
demand_stargazer.covariate_order([
    'net_prices',
    'log_sj_g',
    'is_electric:log_charging_stock_hat',
    'is_electric',
    'log_charging_stock_hat',
    'range',
    'power',
    'battery_capacity'
])
demand_stargazer.rename_covariates({
    'net_prices': 'price',
    'log_sj_g': 'nesting coefficient',
    'is_electric': r'$EV_j$',
    'is_electric:log_charging_stock_hat': r'$(\log(N_m) \times EV_j)$',
    'log_charging_stock_hat': r'$\log(N_m)$',
    'battery_capacity': 'battery capacity'
})
latex_demand = demand_stargazer.render_latex()

# Optionally, save to file
with open('GraphsTables/demand_model_result.tex', 'w', encoding='utf-8') as f:
    f.write(latex_demand)

In [952]:
################ Supply side model

# Create a time trend variable
supply_df['time_trend'] = supply_df['year'].astype(int) - supply_df['year'].astype(int).min() + 1

# Add time_trend
formula = 'log(charging_stations_stock) ~ [log(EV_stock) ~ road_fuel_IV + num_models_in_market + sales_weighted_avg_range] + sub_fix + sub_ope + C(province) + time_trend'

supply_model = IV2SLS.from_formula(formula, data=supply_df).fit()
print(supply_model.summary)

                               IV-2SLS Estimation Summary                               
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9734
Estimator:                              IV-2SLS   Adj. R-squared:                 0.9659
No. Observations:                           155   F-statistic:                 9.753e+05
Date:                          Mon, Aug 04 2025   P-value (F-stat)                0.0000
Time:                                  03:45:21   Distribution:                 chi2(35)
Cov. Estimator:                          robust                                         
                                                                                        
                                      Parameter Estimates                                      
                             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------------


In [953]:
# Export model summary as LaTeX
supply_stargazer = Stargazer([supply_model])
supply_stargazer.covariate_order(['sub_fix', 'sub_ope', 'log(EV_stock)'])
supply_stargazer.rename_covariates({
    'sub_fix': 'fixed subsidy',
    'sub_ope': 'operating subsidy',
    'log(EV_stock)': r'$\lambda_1$'
})
supply_stargazer.significant_digits(5)
latex_supply = supply_stargazer.render_latex()

# Optionally, save to file
with open('GraphsTables/supply_model_result.tex', 'w', encoding='utf-8') as f:
    f.write(latex_supply)

In [990]:
# Save demand model results
with open('demand_model_results.pkl', 'wb') as f:
    pickle.dump(iv_model, f)

# Save supply model results
with open('charging_station_model_results.pkl', 'wb') as f:
    pickle.dump(supply_model, f)

In [955]:
# Export model data for counterfactual analysis
df.to_csv('demand_counterfactual.csv', index=False)
supply_df.to_csv('supply_counterfactual.csv', index=False)

In [991]:
# Demand model results
with open('demand_model_results.pkl', 'rb') as f:
     demand_param = pickle.load(f)

# Charging station model results
with open('charging_station_model_results.pkl', 'rb') as f:
     charging_param = pickle.load(f)

In [957]:
def compute_nested_logit_derivatives(
    df,
    demand_par,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model_id',
    nest_col='nesting_ids',
    within_share_col='within_nest_shares'
):
    """
    Compute derivatives of market shares with respect to prices for each market using equations 12, 14, 15.
    Returns derivatives ∂s_j/∂p_k, not elasticities.
    Returns:
        dict: {market_id: derivative_matrix (DataFrame, index/columns=model_id)}
    """
    alpha = demand_par.params['net_prices']
    sigma = demand_par.params['log_sj_g']
    derivative_matrices = {}

    # Use existing within-nest shares from the dataset
    # No need to recalculate since they're already in within_nest_shares column

    unique_markets = df[market_col].unique()
    for market_id in unique_markets:
        df_mkt = df[df[market_col] == market_id].copy()
        n = len(df_mkt)
        derivatives = np.zeros((n, n))
        shares = df_mkt['shares'].values
        within_nest_shares = df_mkt[within_share_col].values
        nests = df_mkt[nest_col].values
        model_ids = df_mkt[model_col].values

        # Use existing nest_shares for group shares
        group_shares = df_mkt['nest_shares'].values

        for j in range(n):
            s_j = shares[j]
            s_jg = within_nest_shares[j]
            s_g = group_shares[j]
            for k in range(n):
                s_k = shares[k]
                if j == k:
                    # Own-price derivative (Equation 12)
                    deriv = (1/(1-sigma)) * s_j * (1 - sigma*s_jg - (1-sigma)*s_j)
                elif nests[j] == nests[k]:
                    # Cross-price derivative within the same group (Equation 14)
                    deriv = -s_j * s_k * (1 + (sigma/(1-sigma)) * (1/s_g))
                else:
                    # Cross-price derivative across groups (Equation 15)
                    deriv = -s_j * s_k
                # Apply price coefficient to get derivative ∂s_j/∂p_k
                derivatives[j, k] = alpha * deriv

        derivative_df = pd.DataFrame(derivatives, index=model_ids, columns=model_ids)
        derivative_matrices[market_id] = derivative_df

    return derivative_matrices

In [992]:
# Compute derivatives with network effects
derivative_matrices = compute_nested_logit_derivatives(
    df,
    demand_param,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model',
    nest_col='nesting_ids',
    within_share_col='within_nest_shares'
)

In [959]:
# Compute charging station semi-elasticities (γ_j)
def compute_gamma_dict(beta_N, df, nest_col='nesting_ids', market_col='market_ids', share_col='shares', within_share_col='within_nest_shares', sigma=None):
    """
    Compute charging station semi-elasticities (γ_j) for all products, returned as a dict by market.
    Uses existing within-nest shares from the dataset.
    Returns:
        dict: {market_id: gamma vector for that market}
    """
    gamma_dict = {}
    for market_id in df[market_col].unique():
        market_df = df[df[market_col] == market_id].copy()
        # Use existing within-nest shares instead of recalculating
        within_nest_shares = market_df[within_share_col].values
        if isinstance(sigma, dict):
            sigma_g = market_df[nest_col].map(sigma)
        else:
            sigma_g = sigma
        term = (1 / (1 - sigma_g)) * (1 - sigma_g * within_nest_shares - (1 - sigma_g) * market_df[share_col])
        gamma = beta_N * market_df[share_col] * term
        gamma_dict[market_id] = gamma.values
    return gamma_dict

In [ ]:
# Elasticity with network effects
def compute_total_derivatives(df, eta_dict, gamma_dict, v_2, is_ev_col='is_electric', market_col='market_ids'):
    """
    Compute total derivatives of market shares w.r.t. prices (with network effects).
    Args:
        df (pd.DataFrame): DataFrame with all markets.
        eta_dict (dict): {market_id: JxJ eta matrix for that market}.
        gamma_dict (dict): {market_id: gamma vector for that market}.
        v_2 (float): Sensitivity of charging stations to EV sales.
        is_ev_col (str): Column name for EV indicator.
        market_col (str): Column name for market IDs.
    Returns:
        dict: {market_id: JxJ matrix of total derivatives for each market}.
    """
    total_derivatives_dict = {}
    for market_id in df[market_col].unique():
        market_df = df[df[market_col] == market_id].reset_index(drop=True)
        eta = eta_dict[market_id]
        gamma = gamma_dict[market_id]
        is_ev = market_df[is_ev_col].values.astype(bool)
        J = len(gamma)
        total_derivatives = np.zeros((J, J))
        # Use pre-computed EV_share instead of calculating from individual shares
        s_ev = market_df['EV_share'].iloc[0]  # EV_share is the same for all products in a market
        sum_gamma_ev = np.sum(gamma[is_ev])
        eta = np.asarray(eta)  
        for j in range(J):
            for k in range(J):
                if is_ev[j]:
                    feedback_term = 0
                    denom = s_ev - v_2 * sum_gamma_ev
                    if denom != 0:
                        feedback_term = v_2 * gamma[j] * np.sum(eta[is_ev, k]) / denom
                    total_derivatives[j, k] = eta[j, k] + feedback_term
                else:
                    total_derivatives[j, k] = eta[j, k]
        total_derivatives_dict[market_id] = total_derivatives
    return total_derivatives_dict

In [993]:
# Calculate gamma_dict for all markets
beta_N = demand_param.params['log_charging_stock_hat'] + demand_param.params['is_electric:log_charging_stock_hat']
sigma = demand_param.params['log_sj_g']

gamma_dict = compute_gamma_dict(
    beta_N=beta_N,
    df=df,
    nest_col='nesting_ids',
    market_col='market_ids',
    share_col='shares',
    within_share_col='within_nest_shares',
    sigma=sigma
)

# Compute total derivatives using derivative_matrices as eta_dict and gamma_dict as input
total_derivatives_dict = compute_total_derivatives(
    df=df,
    eta_dict=derivative_matrices,
    gamma_dict=gamma_dict,
    v_2=charging_param.params['log(EV_stock)'], 
    is_ev_col='is_electric',
    market_col='market_ids'
)


In [994]:
# Convert derivatives to elasticities - OPTIMIZED VERSION
def derivatives_to_elasticities_fast(derivative_dict, df, price_col='net_prices', market_col='market_ids', model_col='model'):
    """
    Convert derivative matrices to elasticity matrices using vectorized operations.
    Elasticity[j,k] = Derivative[j,k] * (price_k / share_j)
    """
    elasticity_dict = {}
    
    for market_id, derivative_matrix in derivative_dict.items():
        market_df = df[df[market_col] == market_id].copy()
        prices = market_df[price_col].values
        shares = market_df['shares'].values
        
        # Create price and share matrices for vectorized computation
        price_matrix = np.tile(prices, (len(prices), 1))  # Each row has all prices
        share_matrix = np.tile(shares, (len(shares), 1)).T  # Each column has all shares
        
        # Avoid division by zero
        share_matrix = np.where(share_matrix == 0, np.nan, share_matrix)
        
        # Vectorized elasticity calculation
        elasticity_values = derivative_matrix.values * price_matrix / share_matrix
        
        # Replace NaN with 0
        elasticity_values = np.nan_to_num(elasticity_values, nan=0.0)
        
        # Create DataFrame with same index and columns
        elasticity_matrix = pd.DataFrame(
            elasticity_values, 
            index=derivative_matrix.index, 
            columns=derivative_matrix.columns
        )
        
        elasticity_dict[market_id] = elasticity_matrix
    
    return elasticity_dict

print("🚀 Computing elasticities from derivatives (OPTIMIZED)...")
# Compute elasticities from derivatives (without network effects)
elasticity_matrices = derivatives_to_elasticities_fast(
    derivative_dict=derivative_matrices,
    df=df,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model'
)

print("🚀 Computing elasticities from total derivatives (OPTIMIZED)...")
# Compute elasticities from total derivatives (with network effects) - OPTIMIZED
total_elasticity_matrices = {}
for market_id in total_derivatives_dict.keys():
    market_df = df[df['market_ids'] == market_id].copy()
    prices = market_df['net_prices'].values
    shares = market_df['shares'].values
    model_ids = market_df['model'].values
    
    # Convert total derivatives to elasticities using vectorized operations
    total_deriv_matrix = total_derivatives_dict[market_id]
    
    # Create price and share matrices
    price_matrix = np.tile(prices, (len(prices), 1))
    share_matrix = np.tile(shares, (len(shares), 1)).T
    
    # Avoid division by zero
    share_matrix = np.where(share_matrix == 0, np.nan, share_matrix)
    
    # Vectorized computation
    total_elasticity_values = total_deriv_matrix * price_matrix / share_matrix
    total_elasticity_values = np.nan_to_num(total_elasticity_values, nan=0.0)
    
    # Convert to DataFrame with proper index/columns
    total_elasticity_df = pd.DataFrame(
        total_elasticity_values, 
        index=model_ids, 
        columns=model_ids
    )
    total_elasticity_matrices[market_id] = total_elasticity_df

print("✅ COMPLETED elasticity computation:")
print(f"   - Without network effects: {len(elasticity_matrices)} markets")
print(f"   - With network effects: {len(total_elasticity_matrices)} markets")

# Quick performance check
sample_market = list(elasticity_matrices.keys())[0]
print(f"\n📊 Sample results for market {sample_market}:")
print(f"   - Matrix size: {elasticity_matrices[sample_market].shape}")
print(f"   - Own-price elasticities range: [{np.diag(elasticity_matrices[sample_market]).min():.2f}, {np.diag(elasticity_matrices[sample_market]).max():.2f}]")

🚀 Computing elasticities from derivatives (OPTIMIZED)...
🚀 Computing elasticities from total derivatives (OPTIMIZED)...
🚀 Computing elasticities from total derivatives (OPTIMIZED)...
✅ COMPLETED elasticity computation:
   - Without network effects: 155 markets
   - With network effects: 155 markets

📊 Sample results for market P01Y2019:
   - Matrix size: (140, 140)
   - Own-price elasticities range: [-31.36, -1.90]
✅ COMPLETED elasticity computation:
   - Without network effects: 155 markets
   - With network effects: 155 markets

📊 Sample results for market P01Y2019:
   - Matrix size: (140, 140)
   - Own-price elasticities range: [-31.36, -1.90]


In [1005]:
# ===============================================================================
# FIND MARKET WITH POSITIVE DENOMINATORS AND MODELS WITH NEGATIVE NETWORK EFFECTS
# ===============================================================================

print("🔍 ANALYZING MARKETS AND MODELS FOR SAMPLE SELECTION")
print("="*70)

# Get market information from feedback analysis
market_info = feedback_analysis['market_info']

# Find markets with positive denominators
positive_denom_markets = []
for info in market_info:
    if info['denominator'] > 0:
        positive_denom_markets.append({
            'market_id': info['market_id'],
            'denominator': info['denominator'],
            's_ev': info['s_ev'],
            'num_evs': info['num_evs'],
            'num_products': info['num_products']
        })

# Sort by denominator value (highest first for stability)
positive_denom_markets.sort(key=lambda x: x['denominator'], reverse=True)

print(f"📊 Found {len(positive_denom_markets)} markets with positive denominators")
print("\nTop 5 markets with highest positive denominators:")
for i, market in enumerate(positive_denom_markets[:5]):
    print(f"  {i+1}. {market['market_id']}: denominator={market['denominator']:.6f}, "
          f"s_ev={market['s_ev']:.6f}, EVs={market['num_evs']}, products={market['num_products']}")

# Select the market with highest positive denominator and sufficient EVs
selected_market = None
for market in positive_denom_markets:
    if market['num_evs'] >= 4:  # Need at least 4 EVs for a good sample
        selected_market = market
        break

if selected_market is None:
    # If no market has 4+ EVs, take the one with most EVs
    selected_market = max(positive_denom_markets, key=lambda x: x['num_evs'])

print(f"\n✅ SELECTED MARKET: {selected_market['market_id']}")
print(f"   Denominator: {selected_market['denominator']:.6f}")
print(f"   EV market share: {selected_market['s_ev']:.6f}")
print(f"   Number of EVs: {selected_market['num_evs']}")
print(f"   Total products: {selected_market['num_products']}")

# Now find models in this market and analyze their network effects
selected_market_id = selected_market['market_id']
market_df = df[df['market_ids'] == selected_market_id].copy()
ev_models = market_df[market_df['is_electric'] == 1]['model'].unique()

print(f"\n📋 EV MODELS IN SELECTED MARKET ({len(ev_models)} models):")
for model in ev_models:
    print(f"   - {model}")

# Calculate network effects for each model (difference between with and without network effects)
print(f"\n🔬 ANALYZING NETWORK EFFECTS FOR EACH MODEL:")
print("-" * 50)

# Get elasticity matrices for the selected market
elasticity_without = elasticity_matrices[selected_market_id]
elasticity_with = total_elasticity_matrices[selected_market_id]

# Get model names for this market
market_model_names = df[df['market_ids'] == selected_market_id]['model'].values

# Set proper indices for comparison
elasticity_without.index = market_model_names
elasticity_without.columns = market_model_names
elasticity_with.index = market_model_names  
elasticity_with.columns = market_model_names

# Use the same deduplication approach as in cell 18 for consistency
def remove_matrix_duplicates(matrix):
    """Remove duplicate rows and columns from matrix, keeping first occurrence"""
    # Remove duplicate rows
    matrix_no_dup_rows = matrix.loc[~matrix.index.duplicated(keep='first')]
    # Remove duplicate columns
    matrix_clean = matrix_no_dup_rows.loc[:, ~matrix_no_dup_rows.columns.duplicated(keep='first')]
    return matrix_clean

# Apply same deduplication method as used in cell 18
elasticity_without_clean = remove_matrix_duplicates(elasticity_without)
elasticity_with_clean = remove_matrix_duplicates(elasticity_with)

# Calculate network effects (difference in own-price elasticities)
network_effects = {}
for model in ev_models:
    if model in elasticity_without_clean.index:
        try:
            # Use the cleaned matrices to get diagonal values (consistent with cell 18)
            own_elast_without = elasticity_without_clean.loc[model, model]
            own_elast_with = elasticity_with_clean.loc[model, model]
            network_effect = own_elast_with - own_elast_without
            network_effects[model] = {
                'without_network': float(own_elast_without),
                'with_network': float(own_elast_with),
                'network_effect': float(network_effect)
            }
        except Exception as e:
            print(f"Error processing model {model}: {e}")
            continue

# Sort models by network effect (most negative first)
if network_effects:
    sorted_effects = sorted(network_effects.items(), key=lambda x: x[1]['network_effect'])
else:
    print("No network effects calculated successfully.")
    sorted_effects = []

print("Network effects on own-price elasticities:")
for model, effects in sorted_effects:
    print(f"  {model}:")
    print(f"    Without network: {effects['without_network']:.4f}")
    print(f"    With network:    {effects['with_network']:.4f}")
    print(f"    Network effect:  {effects['network_effect']:+.4f} {'(NEGATIVE)' if effects['network_effect'] < 0 else '(POSITIVE)'}")
    print()

# Select models with negative network effects
negative_effect_models = [model for model, effects in sorted_effects 
                         if effects['network_effect'] < 0]

print(f"🎯 MODELS WITH NEGATIVE NETWORK EFFECTS ({len(negative_effect_models)} found):")
for model in negative_effect_models:
    effect = network_effects[model]['network_effect']
    print(f"   - {model}: {effect:+.4f}")

# Select up to 4 models with most negative effects for the sample table
selected_models_for_table = negative_effect_models[:4]

if len(selected_models_for_table) < 4:
    print(f"\n⚠️  Only {len(selected_models_for_table)} models with negative effects found.")
    print("Adding models with least positive effects to reach 4 models...")
    
    positive_effect_models = [model for model, effects in sorted_effects 
                             if effects['network_effect'] >= 0]
    
    # Add models with smallest positive effects
    additional_needed = 4 - len(selected_models_for_table)
    selected_models_for_table.extend(positive_effect_models[:additional_needed])

print(f"\n🎉 FINAL SELECTION FOR SAMPLE TABLE:")
print(f"   Market: {selected_market_id}")
print(f"   Models: {selected_models_for_table}")

# Store the selection for the next cell
SELECTED_SAMPLE_MARKET = selected_market_id
SELECTED_SAMPLE_MODELS = selected_models_for_table

🔍 ANALYZING MARKETS AND MODELS FOR SAMPLE SELECTION
📊 Found 134 markets with positive denominators

Top 5 markets with highest positive denominators:
  1. P07Y2021: denominator=0.004979, s_ev=0.012356, EVs=88, products=239
  2. P07Y2020: denominator=0.004529, s_ev=0.009006, EVs=64, products=189
  3. P04Y2022: denominator=0.003587, s_ev=0.012923, EVs=96, products=215
  4. P05Y2023: denominator=0.003558, s_ev=0.012664, EVs=140, products=281
  5. P04Y2023: denominator=0.003426, s_ev=0.022649, EVs=157, products=297

✅ SELECTED MARKET: P07Y2021
   Denominator: 0.004979
   EV market share: 0.012356
   Number of EVs: 88
   Total products: 239

📋 EV MODELS IN SELECTED MARKET (82 models):
   - Bestune E01
   - Jingyi S50
   - Fengguang E1
   - Fengguang E3
   - Toyota C-HR
   - Yudo π1
   - Yudo π3
   - Wuling Hongguang
   - Wuling Rongguang
   - Lingbao BOX
   - Xuanjie
   - Hycan 007
   - Hycan Z03
   - Jiaji
   - Emgrand
   - Emgrand GSe
   - Xingyue
   - Binyue
   - Venucia D60
   - Venucia

In [1012]:
# Generate sample tables for selected EV models with 6 digits, showing both derivatives and elasticities
# Using market with positive denominators and models with negative network effects

sample_market = SELECTED_SAMPLE_MARKET  # Market with positive denominator
selected_models = ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']  # Models with negative network effects

print(f"📊 GENERATING SAMPLE TABLES")
print(f"Market: {sample_market}")
print(f"Selected models: {selected_models}")
print()

# Ensure only unique model names and present in the selected market
market_df = df[(df['market_ids'] == sample_market) & (df['is_electric'] == 1)]
available_models = [m for m in selected_models if m in market_df['model'].unique()]

print(f"Available models in market: {available_models}")
print()

# Get matrices for the sample market
derivative_matrix_full = derivative_matrices[sample_market]
elasticity_matrix_full = elasticity_matrices[sample_market]
total_elasticity_matrix_full = total_elasticity_matrices[sample_market]

model_names = df[df['market_ids'] == sample_market]['model'].values

# Set index and columns to model names
derivative_matrix_full.index = model_names
derivative_matrix_full.columns = model_names
elasticity_matrix_full.index = model_names
elasticity_matrix_full.columns = model_names
total_elasticity_matrix_full.index = model_names
total_elasticity_matrix_full.columns = model_names

# Create a mask for first occurrence of each model name
_, idx = np.unique(model_names, return_index=True)
unique_model_names = model_names[np.sort(idx)]

# Filter matrices to only include first occurrence for each model name
derivative_matrix = derivative_matrix_full.loc[unique_model_names, unique_model_names]
elasticity_matrix = elasticity_matrix_full.loc[unique_model_names, unique_model_names]
total_elasticity_matrix = total_elasticity_matrix_full.loc[unique_model_names, unique_model_names]

# Use available models as selected (don't filter for uniqueness in model selection)
print(f"Selected models for tables: {available_models}")

# Sample matrices for selected models
sample_derivatives = derivative_matrix.loc[available_models, available_models]
sample_elasticities = elasticity_matrix.loc[available_models, available_models]
sample_total_elasticities = total_elasticity_matrix.loc[available_models, available_models]

# Also get total derivatives with network effects
sample_total_deriv_df = pd.DataFrame(
    total_derivatives_dict[sample_market],
    index=derivative_matrix.index,
    columns=derivative_matrix.columns
).loc[available_models, available_models]

# Remove duplicate rows and columns from the matrices (keep first occurrence)
def remove_matrix_duplicates(matrix):
    """Remove duplicate rows and columns from matrix, keeping first occurrence"""
    # Remove duplicate rows
    matrix_no_dup_rows = matrix.loc[~matrix.index.duplicated(keep='first')]
    # Remove duplicate columns
    matrix_clean = matrix_no_dup_rows.loc[:, ~matrix_no_dup_rows.columns.duplicated(keep='first')]
    return matrix_clean

# Clean all sample matrices to remove duplicates
sample_derivatives = remove_matrix_duplicates(sample_derivatives)
sample_elasticities = remove_matrix_duplicates(sample_elasticities)
sample_total_elasticities = remove_matrix_duplicates(sample_total_elasticities)
sample_total_deriv_df = remove_matrix_duplicates(sample_total_deriv_df)

def matrix_to_latex(matrix, caption):
    latex = "\\begin{table}[!htbp]\n\\centering\n"
    latex += "\\begin{tabular}{@{\\extracolsep{5pt}}l" + "c" * len(matrix.columns) + "}\n"
    latex += "\\\\[-1.8ex]\\hline\n\\\\hline \\\\[-1.8ex]\n"
    latex += "& " + " & ".join(matrix.columns) + " \\\\\n"
    latex += "\\hline \\\\[-1.8ex]\n"
    for idx, row in matrix.iterrows():
        latex += f"{idx} & " + " & ".join([f'{v:.6f}' for v in row]) + " \\\\\n"
    latex += "\\hline \\\\[-1.8ex]\n"
    latex += "\\end{tabular}\n"
    latex += f"\\caption{{{caption}}}\n"
    latex += "\\end{table}\n"
    return latex

# Generate LaTeX tables
latex_derivatives = matrix_to_latex(sample_derivatives, "Derivative matrix (∂s_j/∂p_k) - EV models with negative network effects")
latex_elasticities_no_network = matrix_to_latex(sample_elasticities, "Elasticity matrix WITHOUT network effects - EV models with negative network effects")
latex_elasticities_with_network = matrix_to_latex(sample_total_elasticities, "Elasticity matrix WITH network effects - EV models with negative network effects")

# Save tables
with open('GraphsTables/sample_derivatives_selected.tex', 'w', encoding='utf-8') as f:
    f.write(latex_derivatives)

with open('GraphsTables/sample_elasticity_selected_without_network.tex', 'w', encoding='utf-8') as f:
    f.write(latex_elasticities_no_network)

with open('GraphsTables/sample_elasticity_selected_with_network.tex', 'w', encoding='utf-8') as f:
    f.write(latex_elasticities_with_network)

# Display results
print("\nELASTICITIES WITHOUT NETWORK EFFECTS:")
print("=" * 50)
print(sample_elasticities)
print("\nELASTICITIES WITH NETWORK EFFECTS:")
print("=" * 50)
print(sample_total_elasticities)

# Show the network effects for verification using the same cleaned matrices
print("\n🔍 NETWORK EFFECTS VERIFICATION:")
print("=" * 50)
for model in sample_derivatives.index:
    # Get diagonal values from the cleaned matrices (same as displayed in tables)
    own_elast_without_cleaned = sample_elasticities.loc[model, model]
    own_elast_with_cleaned = sample_total_elasticities.loc[model, model]
    network_effect_cleaned = own_elast_with_cleaned - own_elast_without_cleaned
    
    print(f"{model}:")
    print(f"  Own-price elasticity without network: {own_elast_without_cleaned:.4f}")
    print(f"  Own-price elasticity with network:    {own_elast_with_cleaned:.4f}")
    print(f"  Network effect:                       {network_effect_cleaned:+.4f}")
    print()

📊 GENERATING SAMPLE TABLES
Market: P07Y2021
Selected models: ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']

Available models in market: ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']

Selected models for tables: ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']

ELASTICITIES WITHOUT NETWORK EFFECTS:
             XPeng P5    BYD D1  Toyota C-HR    Kia K3   Qin Pro
XPeng P5    -8.497132  0.081654     0.004500  0.000036  0.000006
BYD D1       0.025155 -8.405643     0.004500  0.000036  0.000006
Toyota C-HR  0.025155  0.081654    -8.599428  0.000036  0.000006
Kia K3       0.000048  0.000156     0.000009 -6.943025  0.003050
Qin Pro      0.000048  0.000156     0.000009  0.018988 -4.798381

ELASTICITIES WITH NETWORK EFFECTS:
             XPeng P5    BYD D1  Toyota C-HR    Kia K3   Qin Pro
XPeng P5    -8.667018 -0.469814    -0.025894 -0.159037 -0.025542
BYD D1      -0.143595 -8.953423    -0.025691 -0.157973 -0.025371
Toyota C-HR -0.145145 -0.471157    -8.629897